In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes sentencepiece pandas scikit-learn nltk evaluate python-docx pdfplumber -q

In [ ]:
import os
import re
import torch
import nltk
import json
import pdfplumber
import pandas as pd
from glob import glob
from tqdm import tqdm

from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForTokenClassification,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    pipeline
)

from peft import LoraConfig, get_peft_model, TaskType

from google.colab import drive, files
from docx import Document

import evaluate
import requests

In [ ]:
drive.mount('/content/drive')

In [ ]:
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

In [ ]:
def preprocess_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [
        word.lower()
        for word in tokens
        if word.isalnum() and word.lower() not in stop_words
    ]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

In [ ]:
print("Loading BERT NER...")

ner_model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
ner_tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

ner_pipeline = pipeline(
    "ner",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple"
)

In [ ]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Model loaded on:", device)

In [ ]:
data_path = "/content/drive/MyDrive/medi_translate.csv"
df = pd.read_csv(data_path)
print("Dataset size:", len(df))
print(df.head())

In [ ]:
target_language = "Tamil / தமிழ்"

df = df[df["Language"] == target_language]

df = df[["Text","Translation"]]
df = df.dropna()

print("Filtered dataset size:", len(df))

In [ ]:
train_texts, val_texts, train_trans, val_trans = train_test_split(
    df["Text"],
    df["Translation"],
    test_size=0.1,
    random_state=42
)

train_df = pd.DataFrame({"src":train_texts,"tgt":train_trans})
val_df = pd.DataFrame({"src":val_texts,"tgt":val_trans})

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [ ]:
source_lang = "eng_Latn"
target_lang = "tam_Taml"

tokenizer.src_lang = source_lang

def preprocess_function(examples):

    inputs = tokenizer(
        examples["src"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    labels = tokenizer(
        examples["tgt"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [ ]:
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/nllb_finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=True
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("/content/finetuned_nllb_medical")
tokenizer.save_pretrained("/content/finetuned_nllb_medical")

In [ ]:
import requests


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

def llama3_simplify(text):
    if not OPENROUTER_API_KEY:
        raise ValueError("Set OPENROUTER_API_KEY in your environment or Colab Secrets before calling this function.")

    prompt = f"""
Convert the hospital discharge summary into a patient-friendly explanation.

STRICT RULES:
1. Every diagnosis MUST include a simple explanation.
2. Every procedure/test MUST explain what the test checks.
3. Every treatment or medicine MUST explain what it is used for.
4. Never leave any medical term unexplained.
5. Use simple everyday language a patient can understand.
6. Always format the output with bullet points and clear sections.
7. Never return the output as a single paragraph.

OUTPUT FORMAT:

Patient Information

Diagnoses
• Disease – simple explanation

Procedures / Tests
• Procedure – what the test checks

Treatment / Medications
• Medicine – what it is used for

Discharge Instructions

DISCHARGE SUMMARY:
{text}
"""

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    body = {
        "model": "meta-llama/llama-3-8b-instruct:nitro",
        "messages": [
            {
                "role": "system",
                "content": """You are a medical assistant explaining hospital discharge reports in simple language.

Rules:
- Explain every diagnosis in simple terms.
- Explain every procedure or test.
- Explain every treatment or medicine and what it is used for.
- Always use bullet points and clear sections.
- Never output a single paragraph."""
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0,
        "top_p": 0.9,
        "max_tokens": 700,
        "frequency_penalty": 0
    }

    response = requests.post(url, headers=headers, json=body)

    data = response.json()

    # Debug print
    print("API Response:", data)

    # Proper error handling
    if "error" in data:
        print("❌ API Error:", data["error"]["message"])
        return None

    if "choices" in data:
        return data["choices"][0]["message"]["content"]

    print("⚠️ Unexpected API response format")
    return None

In [ ]:
print("Upload discharge summary (PDF / DOCX / TXT)")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

if filename.endswith(".txt"):

    with open(filename,"r") as f:
        text = f.read()

elif filename.endswith(".docx"):

    doc = Document(filename)
    text = "\n".join([p.text for p in doc.paragraphs])

elif filename.endswith(".pdf"):

    with pdfplumber.open(filename) as pdf:
        pages = [page.extract_text() for page in pdf.pages]
        text = "\n".join(pages)

print("Document loaded.")

In [ ]:
import nltk
nltk.download("punkt_tab")

In [ ]:
cleaned = preprocess_text(text)
entities = ner_pipeline(cleaned[:512])
simplified = llama3_simplify(cleaned[:2000])

print("\nSIMPLIFIED SUMMARY\n")
print(simplified)

In [ ]:
import re

# ======================================
# Step 1: Prepare structured lines
# ======================================

clean_text = simplified

clean_text = clean_text.replace("**", "")
clean_text = clean_text.replace("*", "")

clean_text = re.sub(r"\n+", "\n", clean_text).strip()

lines = [line.strip() for line in clean_text.split("\n") if line.strip()]


# ======================================
# Step 2: Translation function
# ======================================

def translate_line(text_line):

    tokenizer.src_lang = "eng_Latn"

    inputs = tokenizer(
        text_line,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids("tam_Taml")

    tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=200,
        num_beams=5,
        repetition_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    result = tokenizer.batch_decode(tokens, skip_special_tokens=True)[0]

    result = result.replace("ஃ", ":")

    return result


# ======================================
# Step 3: Translate line by line
# ======================================

translated_lines = []

for line in lines:
    translated_lines.append(translate_line(line))

translation = "\n".join(translated_lines)


# ======================================
# Step 4: Force translate headings
# ======================================

heading_map = {
    "Patient Information": "நோயாளி தகவல்",
    "Diagnoses": "நோயறிதல்",
    "Procedures": "நடைமுறைகள்",
    "Procedures / Tests": "நடைமுறைகள் / சோதனைகள்",
    "Treatment": "சிகிச்சை",
    "Treatment / Medications": "சிகிச்சை / மருந்துகள்",
    "Discharge Instructions": "வெளியேற்ற வழிமுறைகள்"
}

for eng, tam in heading_map.items():
    translation = translation.replace(eng, tam)


# ======================================
# Step 5: Medical terminology corrections
# ======================================

medical_map = {
    "Pneumonia": "நிமோனியா",
    "Bronchitis": "ப்ரோன்கிட்டிஸ்",
    "Hypertension": "உயர் இரத்த அழுத்தம்",
    "Diabetes": "நீரிழிவு",
    "Bronchoscopy": "ப்ரோங்கோஸ்கோபி",
    "Chest X-ray": "மார்பு எக்ஸ்-ரே"
}

for eng, tam in medical_map.items():
    translation = translation.replace(eng, tam)


medicine_map = {
    "Azithromycin": "அசித்ரோமைசின்",
    "Metformin": "மெட்ஃபார்மின்",
    "Amlodipine": "ஆம்லோடிபைன்",
    "Paracetamol": "பாராசிட்டமால்"
}

for eng, tam in medicine_map.items():
    translation = translation.replace(eng, tam)


# ======================================
# Step 6: Fix Tamil word issues
# ======================================

translation = translation.replace("சிகிச்சை\nயளிக்கப்பட்டு", "சிகிச்சையளிக்கப்பட்டு")

translation = translation.replace(
    "முதன்மை பராமரிப்பு மருத்துவர்: Rajesh Malhotra, MD",
    "முதன்மை பராமரிப்பு மருத்துவர்: ராஜேஷ் மால்ஹோத்ரா, MD"
)

translation = translation.replace("மெட்:போர்மின்", "மெட்ஃபார்மின்")
translation = translation.replace(" MD", " எம்.டி")
translation = translation.replace("mg daily", " mg தினசரி")


# ======================================
# Step 7: Formatting
# ======================================

translation = re.sub(r"(நோயாளி தகவல்)", r"\n\n\1\n", translation)
translation = re.sub(r"(நோயறிதல்)", r"\n\n\1\n", translation)
translation = re.sub(r"(நடைமுறைகள் / சோதனைகள்)", r"\n\n\1\n", translation)
translation = re.sub(r"(சிகிச்சை / மருந்துகள்)", r"\n\n\1\n", translation)
translation = re.sub(r"(வெளியேற்ற வழிமுறைகள்)", r"\n\n\1\n", translation)

translation = re.sub(r"(நிமோனியா)", r"\n• \1", translation)
translation = re.sub(r"(ப்ரோன்கிட்டிஸ்)", r"\n• \1", translation)
translation = re.sub(r"(உயர் இரத்த அழுத்தம்)", r"\n• \1", translation)
translation = re.sub(r"(நீரிழிவு)", r"\n• \1", translation)

translation = re.sub(r"(அசித்ரோமைசின்)", r"\n• \1", translation)
translation = re.sub(r"(மெட்ஃபார்மின்)", r"\n• \1", translation)
translation = re.sub(r"(ஆம்லோடிபைன்)", r"\n• \1", translation)
translation = re.sub(r"(பாராசிட்டமால்)", r"\n• \1", translation)

translation = re.sub(r"(உங்கள்)", r"\n• \1", translation)


# ======================================
# Step 8: FINAL CLEANUP
# ======================================

# remove duplicate discharge headings
translation = re.sub(
    r"(வெளியேற்ற வழிமுறைகள்)\s*\n\s*\1",
    r"\1",
    translation
)

# fix diabetes line
translation = translation.replace(
    "வகை 2 \n• நீரிழிவு நோய்",
    "• வகை 2 நீரிழிவு நோய்"
)

# ensure every instruction line has bullet
lines = translation.split("\n")
fixed = []

for line in lines:

    if (
        "பராமரிக்கவும்" in line
        or "தவிர்க்கவும்" in line
        or "திட்டமிடுங்கள்" in line
        or "கண்காணிக்கவும்" in line
        or "தொடரவும்" in line
    ):
        if not line.strip().startswith("•"):
            line = "• " + line.strip()

    fixed.append(line)

translation = "\n".join(fixed)

translation = re.sub(r"\n+", "\n", translation).strip()


# ======================================
# Step 9: Output
# ======================================

print("\nFINAL TRANSLATED OUTPUT\n")
print(translation)

In [ ]:
import evaluate
import json
import re

print("\n🔍 Evaluating Translation Quality...\n")

# ===== Normalization Function =====
def normalize(text):

    text = text.lower()

    # remove bullets
    text = text.replace("•", " ")

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ===== Reference Translation =====
reference_translation = input("Enter the reference translation in Tamil: ").strip()

prediction = translation.strip()
reference = reference_translation.strip()

# ===== Normalize texts =====
prediction_norm = normalize(prediction)
reference_norm = normalize(reference)

# ===== Load metrics =====
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")

# ===== Compute metrics =====
rouge_score = rouge.compute(
    predictions=[prediction_norm],
    references=[reference_norm]
)

bleu_score = bleu.compute(
    predictions=[prediction_norm],
    references=[[reference_norm]]
)

meteor_score = meteor.compute(
    predictions=[prediction_norm],
    references=[reference_norm]
)

# ===== Additional Metrics =====
pred_words = prediction_norm.split()
ref_words = reference_norm.split()

prediction_length = len(pred_words)
reference_length = len(ref_words)

length_ratio = prediction_length / reference_length if reference_length > 0 else 0

char_length_ratio = len(prediction_norm) / len(reference_norm) if len(reference_norm) > 0 else 0

pred_chars = set(prediction_norm)
ref_chars = set(reference_norm)

char_overlap = len(pred_chars & ref_chars) / len(pred_chars | ref_chars) if len(pred_chars | ref_chars) > 0 else 0

# ===== Print results =====
print("🔹 ROUGE Scores")
print(f"ROUGE-1: {rouge_score['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_score['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_score['rougeL']:.4f}")

print("\n🔹 BLEU Score")
print(f"BLEU: {bleu_score['bleu']:.4f}")

print("\n🔹 METEOR Score")
print(f"METEOR: {meteor_score['meteor']:.4f}")

print("\n🔹 Additional Metrics")
print(f"Prediction Length: {prediction_length}")
print(f"Reference Length: {reference_length}")
print(f"Length Ratio: {length_ratio:.4f}")
print(f"Char Length Ratio: {char_length_ratio:.4f}")
print(f"Character Overlap: {char_overlap:.4f}")

# ===== Save results =====
results = {
    "rouge1": rouge_score["rouge1"],
    "rouge2": rouge_score["rouge2"],
    "rougeL": rouge_score["rougeL"],
    "bleu": bleu_score["bleu"],
    "meteor": meteor_score["meteor"],
    "prediction_length": prediction_length,
    "reference_length": reference_length,
    "length_ratio": length_ratio,
    "char_length_ratio": char_length_ratio,
    "char_overlap": char_overlap
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\n✅ Metrics saved to evaluation_results.json")

In [ ]:
!pip install rouge_score
!pip install evaluate
!pip install sacrebleu
!pip install nltk